# Custom solvers
In this example, we show how to define custom solvers. Our system
will again be silicon, because we are not very imaginative

In [1]:
using DFTK, LinearAlgebra

a = 10.26
lattice = a / 2 * [[0 1 1.];
                   [1 0 1.];
                   [1 1 0.]]
Si = ElementPsp(:Si, load_psp("hgh/lda/Si-q4"))
atoms = [Si, Si]
positions =  [ones(3)/8, -ones(3)/8]

# We take very (very) crude parameters
model = model_DFT(lattice, atoms, positions; functionals=LDA())
basis = PlaneWaveBasis(model; Ecut=5, kgrid=[1, 1, 1]);

We define our custom fix-point solver: simply a damped fixed-point

In [2]:
function my_fp_solver(f, x0, info0; maxiter)
    mixing_factor = .7
    x = x0
    info = info0
    for n = 1:maxiter
        fx, info = f(x, info)
        if info.converged || info.timedout
            break
        end
        x = x + mixing_factor * (fx - x)
    end
    (; fixpoint=x, info)
end;

Note that the fixpoint map `f` operates on an auxiliary variable `info` for
state bookkeeping. Early termination criteria are flagged from inside
the function `f` using boolean flags `info.converged` and `info.timedout`.
For control over these criteria, see the `is_converged` and `maxtime`
keyword arguments of `self_consistent_field`.

Our eigenvalue solver just forms the dense matrix and diagonalizes
it explicitly (this only works for very small systems)

In [3]:
function my_eig_solver(A, X0; maxiter, tol, kwargs...)
    n = size(X0, 2)
    A = Array(A)
    E = eigen(A)
    λ = E.values[1:n]
    X = E.vectors[:, 1:n]
    (; λ, X, residual_norms=[], n_iter=0, converged=true, n_matvec=0)
end;

Finally we also define our custom mixing scheme. It will be a mixture
of simple mixing (for the first 2 steps) and than default to Kerker mixing.
In the mixing interface `δF` is $(ρ_\text{out} - ρ_\text{in})$, i.e.
the difference in density between two subsequent SCF steps and the `mix`
function returns $δρ$, which is added to $ρ_\text{in}$ to yield $ρ_\text{next}$,
the density for the next SCF step.

In [4]:
struct MyMixing
    n_simple  # Number of iterations for simple mixing
end
MyMixing() = MyMixing(2)

function DFTK.mix_density(mixing::MyMixing, basis, δF; n_iter, kwargs...)
    if n_iter <= mixing.n_simple
        return δF  # Simple mixing -> Do not modify update at all
    else
        # Use the default KerkerMixing from DFTK
        DFTK.mix_density(KerkerMixing(), basis, δF; kwargs...)
    end
end

That's it! Now we just run the SCF with these solvers

In [5]:
scfres = self_consistent_field(basis;
                               tol=1e-4,
                               solver=my_fp_solver,
                               eigensolver=my_eig_solver,
                               mixing=MyMixing());

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -7.219088824507                   -0.48    0.0    1.55s
  2   -7.247015924824       -1.55       -0.86    0.0    1.23s
  3   -7.251014664290       -2.40       -1.30    0.0    122ms
  4   -7.251260331727       -3.61       -1.61    0.0   71.9ms
  5   -7.251319377652       -4.23       -1.91    0.0    108ms
  6   -7.251333778615       -4.84       -2.21    0.0   69.6ms
  7   -7.251337429080       -5.44       -2.50    0.0   70.1ms
  8   -7.251338403313       -6.01       -2.78    0.0   90.0ms
  9   -7.251338678397       -6.56       -3.05    0.0   70.9ms
 10   -7.251338760424       -7.09       -3.31    0.0   79.3ms
 11   -7.251338786085       -7.59       -3.57    0.0   70.3ms
 12   -7.251338794432       -8.08       -3.82    0.0   75.5ms
 13   -7.251338797230       -8.55       -4.06    0.0   83.3ms


Note that the default convergence criterion is the difference in
density. When this gets below `tol`, the fixed-point solver terminates.
You can also customize this with the `is_converged` keyword argument to
`self_consistent_field`, as shown below.

## Customizing the convergence criterion
Here is an example of a defining a custom convergence criterion and specifying
it using the `is_converged` callback keyword to `self_consistent_field`.

In [6]:
function my_convergence_criterion(info)
    tol = 1e-10
    length(info.history_Etot) < 2 && return false
    ΔE = (info.history_Etot[end-1] - info.history_Etot[end])
    ΔE < tol
end

scfres2 = self_consistent_field(basis;
                                solver=my_fp_solver,
                                is_converged=my_convergence_criterion,
                                eigensolver=my_eig_solver,
                                mixing=MyMixing());

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -7.219088824507                   -0.48    0.0    156ms
  2   -7.247015924824       -1.55       -0.86    0.0    136ms
  3   -7.251014664290       -2.40       -1.30    0.0   72.3ms
  4   -7.251260331727       -3.61       -1.61    0.0   98.3ms
  5   -7.251319377652       -4.23       -1.91    0.0   70.2ms
  6   -7.251333778615       -4.84       -2.21    0.0   74.5ms
  7   -7.251337429080       -5.44       -2.50    0.0   92.8ms
  8   -7.251338403313       -6.01       -2.78    0.0   71.9ms
  9   -7.251338678397       -6.56       -3.05    0.0   94.9ms
 10   -7.251338760424       -7.09       -3.31    0.0   69.9ms
 11   -7.251338786085       -7.59       -3.57    0.0   76.9ms
 12   -7.251338794432       -8.08       -3.82    0.0   70.5ms
 13   -7.251338797230       -8.55       -4.06    0.0   75.3ms
 14   -7.251338798189       -9.02       -4.30    0.0   75.1ms
 15   -7.